In [1]:
# Import Libraries
import math
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt

In [2]:
# Seaborn Styling
sns.set()
sns.set_context("poster", font_scale = 1.25)
sns.set_style("ticks")

In [6]:
# Experiment constants
# To run with a subset of experiments, edit the lists below to match your data:
#   - Only BFS (no MST): set "pypy": ['bfs'], and optionally "jvm": [] to skip Java
#   - Only rate 20: set eviction_rates = [20]
#   - Only certain strategies: set strategies to the subset you ran, e.g. ['cold', 'request_centric&max_capacity=12']
modes = ['once-per-minute', 'once-per-five-minutes', 'once-per-hour']
platforms = {
            #  "pypy": ['bfs', 'compress', 'dfs', 'mst', 'dynamic-html', 'pagerank'],
             "pypy": ['bfs','dfs', 'mst', 'dynamic-html', 'pagerank', 'compress', 'upload', 'thumbnail', 'video'],
             "jvm": ['matrix-multiplication', 'word-count', 'simple-hash', 'html-rendering']
            }
strategies = ['cold', 'fixed&request_to_checkpoint=1', 'request_centric&max_capacity=12' ]
eviction_rates = [1, 4, 20]
mutabilities = [1]
df_columns = ['request_number', 'benchmark', 'mutability', 'strategy', 'rate', 'client', 'server', 'overhead']

In [ ]:
# Load data (safe: missing files give empty DataFrame so rest of notebook can still run)
def load_csv_safe(path: str):
    try:
        return pd.read_csv(path, names=df_columns)
    except FileNotFoundError:
        print(f"Warning: {path} not found. Using empty data.")
        return pd.DataFrame(columns=df_columns)

python_df = load_csv_safe('../data/python-evaluation.csv')
java_df = load_csv_safe('../data/java-evaluation.csv')
# Show what benchmarks, rates, and strategies are present (helps when using a subset)
for name, df in [('Python', python_df), ('Java', java_df)]:
    if not df.empty:
        print(f"{name}: benchmarks={sorted(df['benchmark'].unique())}, rates={sorted(df['rate'].unique())}, strategies={sorted(df['strategy'].unique())}")
    else:
        print(f"{name}: no data")

In [7]:
def convergence(df: pd.DataFrame, platform: str, benchmark: str, eviction_rate: np.int64):
    """Uses pre-loaded df (python_df or java_df). Returns None if no data."""
    # Extract data only for the request centric strategy
    df = df[df['strategy'] == 'request_centric&max_capacity=12']
    df = df[df['rate'] == eviction_rate]
    df = df[(df['benchmark'] == benchmark)]
    if df.empty or len(df) < 100:
        return None
    latencies = df['client'].to_numpy()
    tail_start = int(0.8 * 500)
    target = np.median(df[df['request_number'] >= tail_start]['client'].to_numpy())
    target_l, target_h = target * 0.98, target * 1.02
    window_size = 20
    skip = 100 if platform == "pypy" else 200
    for index in range(skip, len(latencies)):
        window = latencies[index : index + window_size]
        if len(window) < window_size:
            break
        if target_l <= np.median(window) <= target_h:
            return index
    return None

In [8]:
table = {}
df_by_platform = {"pypy": python_df, "jvm": java_df}
for platform in platforms:
    df = df_by_platform[platform]
    table[platform] = {}
    for benchmark in platforms[platform]:
        table[platform][benchmark] = {}
        for mutability in mutabilities:
            table[platform][benchmark][mutability] = {}
            for eviction_rate in eviction_rates:
                table[platform][benchmark][mutability][eviction_rate] = convergence(df, platform, benchmark, eviction_rate)
for platform in platforms:
    for benchmark in platforms[platform]:
        for mutability in mutabilities:
            for eviction_rate in eviction_rates:
                if eviction_rate == 4:
                    print(f"{benchmark}: {table[platform][benchmark][mutability][eviction_rate]}")

bfs: 113
dfs: 113
mst: 135
dynamic-html: 210
pagerank: 126
compress: 100
upload: 144
thumbnail: 100
video: 165
matrix-multiplication: 202
word-count: 213
simple-hash: 201
html-rendering: 203


# Performance Numbers (CDFs)

In [9]:
import pandas as pd

In [10]:
function_titles = {
"bfs": 'BFS',
"dfs": 'DFS',
"dynamic-html": 'DynamicHTML',
"mst": 'MST',
"pagerank": 'PageRank',
"compress": 'Compression',
"upload": 'Uploader',
"thumbnail": 'Thumbnailer',
"video":'Video',
"matrix-multiplication": 'MatrixMult',
"simple-hash": 'Hash',
"html-rendering": 'HTML Rendering',
"word-count": 'WordCount',
}
platforms = ["python", "java"]
eviction_rates = [1, 4, 20]
strategies = ['cold', 'fixed&request_to_checkpoint=1', 'request_centric&max_capacity=12' ]

## Orchestration Strategy

In [16]:
# Use data loaded in the "Load data" cell above
df = pd.concat([python_df, java_df])
df = df[df['rate'] == 1]
# For improvement stats we need both baseline and request_centric in the data
required = ['fixed&request_to_checkpoint=1', 'request_centric&max_capacity=12']
missing = [s for s in required if s not in df['strategy'].unique()]
if missing:
    print(f"Note: missing strategies in data: {missing}. Some cells below may fail or show NaNs.")

In [17]:
df_grouped = df.groupby(["benchmark", "strategy"]).median()["client"].reset_index()
df_pivot = df_grouped.pivot(index='benchmark', columns='strategy', values='client')

baseline_col = 'fixed&request_to_checkpoint=1'
eval_col = 'request_centric&max_capacity=12'
if baseline_col not in df_pivot.columns or eval_col not in df_pivot.columns:
    print("Skipping improvement: need both 'fixed&request_to_checkpoint=1' and 'request_centric&max_capacity=12' in data.")
else:
    df_pivot['improvement'] = (df_pivot[baseline_col] - df_pivot[eval_col]) / df_pivot[baseline_col] * 100
    df_pivot_positive = df_pivot[df_pivot['improvement'] > 5]
    if not df_pivot_positive.empty:
        min_positive_improvement_benchmark = df_pivot_positive['improvement'].idxmin()
        print(f"The benchmark with the minimum positive improvement is: {min_positive_improvement_benchmark}")
    max_improvement_benchmark = df_pivot['improvement'].idxmax()
    print(f"The benchmark with the maximum improvement is: {max_improvement_benchmark}")
    print(df_pivot['improvement'])

The benchmark with the minimum positive improvement is: compress
The benchmark with the maximum improvement is: simple-hash
benchmark
bfs                      24.957781
compress                  7.525384
dfs                      48.870966
dynamic-html              4.443416
html-rendering           55.755759
matrix-multiplication    13.745837
mst                      45.654389
pagerank                 14.634416
simple-hash              60.221936
thumbnail               -19.721264
upload                  -23.354426
video                    -2.645932
word-count               50.232988
Name: improvement, dtype: float64


In [19]:
# Geometric Mean

from scipy.stats import gmean

# Use benchmarks that are present in the data (works for subset runs)
benchmarks_list = sorted(df["benchmark"].unique().tolist())
print(f"Number of benchmarks: {len(benchmarks_list)}")

# Filter df for these benchmarks
df_filtered = df[df["benchmark"].isin(benchmarks_list)]

# Calculate the median overhead for each benchmark and strategy
df_grouped = df_filtered.groupby(["benchmark", "strategy"]).median()["client"].reset_index()

# Pivot the table to have each strategy as a separate column
df_pivot = df_grouped.pivot(index='benchmark', columns='strategy', values='client')

baseline_col = 'fixed&request_to_checkpoint=1'
eval_col = 'request_centric&max_capacity=12'
if baseline_col not in df_pivot.columns or eval_col not in df_pivot.columns:
    print("Skipping geometric mean: need both fixed and request_centric strategy columns in data.")
else:
    df_pivot['improvement'] = (df_pivot[baseline_col] - df_pivot[eval_col]) / df_pivot[baseline_col] * 100
    threshold_positive = 5.0
    threshold_negative = -5.0
    improved_benchmarks = df_pivot[df_pivot['improvement'] > threshold_positive].index
    on_par_benchmarks = df_pivot[(df_pivot['improvement'] >= threshold_negative) & (df_pivot['improvement'] <= threshold_positive)].index
    worsened_benchmarks = df_pivot[df_pivot['improvement'] < threshold_negative].index
    print(f"Benchmarks that improved: {improved_benchmarks.tolist()} ({len(improved_benchmarks)})")
    print(f"On par: {on_par_benchmarks.tolist()} ({len(on_par_benchmarks)})")
    print(f"Worsened: {worsened_benchmarks.tolist()} ({len(worsened_benchmarks)})")
    df_pivot_positive = df_pivot[df_pivot['improvement'] > threshold_positive]
    if len(df_pivot_positive) > 0:
        geo_mean_improvement = gmean(df_pivot_positive['improvement'])
        print(f"Geometric mean of positive improvements: {geo_mean_improvement:.2f}%")


Number of benchmarks: 13
Benchmarks (and corresponding rates) that improved: ['bfs', 'compress', 'dfs', 'html-rendering', 'matrix-multiplication', 'mst', 'pagerank', 'simple-hash', 'word-count']
Number of benchmarks that improved: 9
Benchmarks (and corresponding rates) that are on par: ['dynamic-html', 'video']
Number of benchmarks that are on par: 2
Benchmarks (and corresponding rates) that worsened: ['thumbnail', 'upload']
Number of benchmarks that worsened: 2
The geometric mean of the median percentage improvements for the benchmarks with positive improvements is: 28.94%


## Request Rates

In [20]:
# Use data loaded in the "Load data" cell above
df = pd.concat([python_df, java_df])

In [21]:
from scipy.stats import gmean

# Calculate the median client times for each benchmark, strategy, and rate
df_grouped = df.groupby(["benchmark", "strategy", "rate"]).median()["client"].reset_index()

# Pivot the table to have each strategy as a separate column, while maintaining benchmark and rate in the index
df_pivot = df_grouped.pivot(index=['benchmark', 'rate'], columns='strategy', values='client')

baseline_col = 'fixed&request_to_checkpoint=1'
eval_col = 'request_centric&max_capacity=12'
if baseline_col not in df_pivot.columns or eval_col not in df_pivot.columns:
    print("Skipping per-rate improvement: need both fixed and request_centric in data.")
else:
    df_pivot['improvement'] = (df_pivot[baseline_col] - df_pivot[eval_col]) / df_pivot[baseline_col] * 100
    df_pivot_positive = df_pivot[df_pivot['improvement'] > 5].reset_index()
    if not df_pivot_positive.empty:
        geo_mean_improvement_per_rate = df_pivot_positive.groupby("rate")['improvement'].apply(gmean)
        print(f"The geometric mean of the median improvements per rate is:\n{geo_mean_improvement_per_rate}")
    else:
        print("No (benchmark, rate) pairs with >5% improvement.")

The geometric mean of the median improvements per rate is:
rate
1     28.935272
4     21.320460
20    16.622072
Name: improvement, dtype: float64


In [22]:
from scipy.stats import gmean

# Calculate the median client times for each benchmark, strategy, and rate
df_grouped = df.groupby(["benchmark", "strategy", "rate"]).median()["client"].reset_index()

# Pivot the table to have each strategy as a separate column, while maintaining benchmark and rate in the index
df_pivot = df_grouped.pivot(index=['benchmark', 'rate'], columns='strategy', values='client')

baseline_col = 'fixed&request_to_checkpoint=1'
eval_col = 'request_centric&max_capacity=12'
if baseline_col not in df_pivot.columns or eval_col not in df_pivot.columns:
    print("Skipping: need both fixed and request_centric in data.")
else:
    df_pivot['improvement'] = (df_pivot[baseline_col] - df_pivot[eval_col]) / df_pivot[baseline_col] * 100
    threshold_positive, threshold_negative = 5.0, -5.0
    improved_benchmarks = df_pivot[df_pivot['improvement'] > threshold_positive].index
    on_par_benchmarks = df_pivot[(df_pivot['improvement'] >= threshold_negative) & (df_pivot['improvement'] <= threshold_positive)].index
    worsened_benchmarks = df_pivot[df_pivot['improvement'] < threshold_negative].index
    print(f"Improved: {improved_benchmarks.tolist()} ({len(improved_benchmarks)})")
    print(f"On par: {on_par_benchmarks.tolist()} ({len(on_par_benchmarks)})")
    print(f"Worsened: {worsened_benchmarks.tolist()} ({len(worsened_benchmarks)})")
    df_pivot_positive = df_pivot[df_pivot['improvement'] > threshold_positive]
    if len(df_pivot_positive) > 0:
        print(f"Geometric mean of positive improvements: {gmean(df_pivot_positive['improvement']):.2f}%")

print(f"The geometric mean of the median percentage improvements for the benchmarks with positive improvements is: {geo_mean_improvement:.2f}%")

Benchmarks (and corresponding rates) that improved: [('bfs', 1), ('bfs', 4), ('bfs', 20), ('compress', 1), ('compress', 4), ('dfs', 1), ('dfs', 4), ('dfs', 20), ('dynamic-html', 4), ('dynamic-html', 20), ('html-rendering', 1), ('html-rendering', 4), ('html-rendering', 20), ('matrix-multiplication', 1), ('matrix-multiplication', 4), ('matrix-multiplication', 20), ('mst', 1), ('mst', 4), ('mst', 20), ('pagerank', 1), ('pagerank', 4), ('pagerank', 20), ('simple-hash', 1), ('simple-hash', 4), ('simple-hash', 20), ('thumbnail', 20), ('upload', 20), ('word-count', 1), ('word-count', 4), ('word-count', 20)]
Number of benchmarks that improved: 30
Benchmarks (and corresponding rates) that are on par: [('compress', 20), ('dynamic-html', 1), ('thumbnail', 4), ('video', 1), ('video', 20)]
Number of benchmarks that are on par: 5
Benchmarks (and corresponding rates) that worsened: [('thumbnail', 1), ('upload', 1), ('upload', 4), ('video', 4)]
Number of benchmarks that worsened: 4
The geometric mean 